# 3D penalty contact in LowLevelFEM

This short example demonstrates frictionless three-dimensional contact with the
penalty method.

Two elastic blocks are initially separated by a small gap. A prescribed
non-uniform displacement bends the upper contact surface downward until contact
develops near the center.

The total prescribed displacement is applied in three equal load steps. At each
step a simple relaxed contact iteration is used.

For normal penalty contact,

$$
K_c =
\int_{\Gamma_c}
G^T c_n G \, d\Gamma ,
$$

where `G = ContactGap(C)` is evaluated directly at slave-side Gauss points.

The contact residual is

$$
r_c = K_c (r+u),
$$

and the correction follows from

$$
(K+K_c)\Delta u = -R.
$$


In [27]:
using LowLevelFEM, LinearAlgebra, IterativeSolvers, Preconditioners

openGeometry("boxes.geo")


## Model

Both bodies use the same three-dimensional linear elastic material.


In [28]:
mat = Material("body")

U = Field(
    [mat],
    type=:VectorField,
    dim=3,
    fieldName=:u
)

K = ∫(
    SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U)
)

# Zero external load.
f = ∫(U ⋅ [0, 0, 0])


nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

## Initial state

The lower body is fixed. The upper boundary is fixed in the horizontal
directions, while its vertical displacement will be prescribed incrementally.

The final displacement profile is

$$
u_y(x,z)
=
-\frac{x(x-10)z(z-10)}{4250}.
$$

It is zero near the outer edges and largest at the center, so the upper body
gradually comes into contact with the lower body.


In [29]:
support0 = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]

u0 = solveField(
    K,
    f,
    support=support0
)

showDoFResults(
    u0,
    name="initial displacement",
    factor=1,
    visible=false
)


0

## Contact pair

The `Contact` object stores the slave-master geometry and closest-point search
data. `ContactGap(C)` creates the normal contact operator used directly in the
weak form.


In [30]:
C = contact(
    U,
    master="master",
    slave="slave",
    displacement=u0,
    topology_tol=0.01
)

G = ContactGap(C)

# Absolute nodal reference positions.
r = nodePositionVector(U)

# Normal penalty stiffness.
cn = 1e6


1.0e6

## Incremental boundary conditions

The total displacement is divided into three identical increments. Nonlinear
contact corrections use homogeneous boundary conditions so that the prescribed
displacement level remains unchanged during the iteration.


In [31]:
nsteps = 3

uy_step(x, y, z) =
    -x * (x - 10) * z * (z - 10) /
    (4250 * nsteps)

support_load_step = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uz=0, uy=uy_step)
]

support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]

free = freeDoFs(U, support_increment)

freeNorm(v::VectorField) =
    LinearAlgebra.norm(
        elementsToNodes(v).a[free, 1]
    )


freeNorm (generic function with 1 method)

## Three load steps

Each load step first applies one elastic displacement increment. The contact
equilibrium is then corrected iteratively.

The nonlinear loop contains only four essential operations:

1. assemble the current contact tangent,
2. compute the residual,
3. solve for the displacement correction,
4. update the displacement.

The contact geometry is updated internally by `updateFrom=u_it`.


In [35]:
maxiter = 20
tol = 1e-3
ω = 0.5

u = copy(u0)

for step in 1:nsteps

    println()
    println("Load step ", step, "/", nsteps)

    # Elastic predictor for one displacement increment.
    Δu_load = solveField(
        Symmetric(K),
        f,
        support=support_load_step,
        solver=:cg
    )

    u_it = u + Δu_load

    for iter in 1:maxiter

        # Penalty contact tangent on the current configuration.
        Kc = ∫(
            G ⋅ cn ⋅ G;
            updateFrom=u_it,
            gauss=2,
            threads=:auto
        )

        # Total residual.
        rc = Kc * (r + u_it)
        R = K * u_it - f + rc

        # Contact correction with homogeneous essential BCs.
        Δu = solveField(
            Symmetric(K + Kc),
            -R,
            support=support_increment,
            solver=:cg,
            preconditioner=:ich,
            solveroptions=(reltol=1e-4,)
        )

        err =
            freeNorm(Δu) /
            max(
                freeNorm(u_it),
                eps(Float64)
            )

        println(
            "  iter = ", iter,
            ", error = ", err
        )

        # Relaxed update.
        u_it = u_it + ω * Δu

        err < tol && break
    end

    u = u_it
end



Load step 1/3
  iter = 1, error = 7.16958581936632e-8

Load step 2/3
  iter = 1, error = 5.377181284259913e-8

Load step 3/3
  iter = 1, error = 0.044835264480324676
  iter = 2, error = 0.022659085256214986
  iter = 3, error = 0.011469702532788638
  iter = 4, error = 0.005818418166963393
  iter = 5, error = 0.002993828116999068
  iter = 6, error = 0.001542994012974793
  iter = 7, error = 0.0007887297489624513


## Results

The complete nodal contact state is synchronized once after the final load
step. The normal gap is then evaluated from the same contact geometry.

With the convention `gap < 0` in penetration, the penalty pressure is

$$
p_n = \max(-c_n g_n,0).
$$


In [36]:
updateContact!(C, u)

gap = ContactGap(
    C,
    u;
    gauss=4
)

pressure = mapScalarField(
    g -> max(-cn * g, 0.0),
    gap
)

showDoFResults(
    u,
    name="displacement",
    factor=1,
    visible=false
)

showElementResults(
    nodesToElements(
        gap,
        onPhysicalGroup="slave"
    ),
    name="gap"
)

showElementResults(
    elementsToElements(
        pressure,
        onPhysicalGroup="slave"
    ),
    name="contact pressure"
)

openPostProcessor()


## Notes

This example intentionally uses a very simple nonlinear algorithm.

- The load is displacement controlled.
- The three load steps are uniform.
- No line search or adaptive cutback is used.
- `updateFrom=u_it` performs the lightweight geometry update required by the
  Gauss-point contact integration.
- A larger penalty stiffness reduces penetration, but excessive values can
  deteriorate conditioning and may produce mesh-scale pressure oscillations.

The example is therefore intended as a compact introduction to the
three-dimensional `ContactGap` penalty formulation rather than a general
nonlinear contact solver.
